# PSGO — Full Paper Pipeline (29 Functions × 30 Runs)
**CEC2017 Full Protocol** | D=30 | FES=10,000×D | 30 runs per algo

### Steps:
1. Runtime → Change runtime type → **T4 GPU**
2. Run cells top to bottom — checkpoint har step pe save karta hai
3. Estimated time: **~2-3 hours** on Colab T4


In [1]:
# Cell 1 — Drive mount + ZIP extract + helpers
import os, shutil, zipfile, json
from google.colab import drive, files

drive.mount('/content/drive')

os.makedirs('/content/results', exist_ok=True)
os.makedirs('/content/paper_outputs', exist_ok=True)
DRIVE_PATH = '/content/drive/MyDrive/PSGO_Results_Full'
os.makedirs(DRIVE_PATH, exist_ok=True)

def save_to_drive(fname, subdir='results'):
    src = f'/content/{subdir}/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, f'{DRIVE_PATH}/{fname}')
        print(f'  Saved: {fname}')

def restore_from_drive(fname):
    src = f'{DRIVE_PATH}/{fname}'
    dst = f'/content/results/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'  Restored: {fname}')
    return os.path.exists(dst)

# ZIP path — Drive se pehle check karo
ZIP_DRIVE = '/content/drive/MyDrive/PSGO_all_files.zip'
ZIP_LOCAL = '/content/PSGO_all_files.zip'

if os.path.exists(ZIP_DRIVE):
    zip_path = ZIP_DRIVE
    print('ZIP found in Drive')
elif os.path.exists(ZIP_LOCAL):
    zip_path = ZIP_LOCAL
    print('ZIP found locally')
else:
    print('Upload PSGO_all_files.zip:')
    up = files.upload()
    zip_path = '/content/' + list(up.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/')
print('Files:', [f for f in os.listdir('/content/') if f.endswith('.py')])

# Restore previous results
restored = [f for f in ['cec2017_full.json','feature_selection.json',
                         'engineering.json','stats.json']
            if restore_from_drive(f)]
print(f'Restored: {restored if restored else "nothing (fresh start)"}')
print('Setup COMPLETE!')


Mounted at /content/drive
Upload PSGO_all_files.zip:


Saving PSGO_all_files_v4.zip to PSGO_all_files_v4.zip
Files: ['psgo.py', 'run_all_fast.py', 'run_cec2017.py', 'generate_paper_outputs.py', 'competitors.py']
  Restored: cec2017_full.json
Restored: ['cec2017_full.json']
Setup COMPLETE!


In [2]:
# Cell 2 — Install packages (fixed)
import subprocess, sys
print("Installing...")

# opfunu version fix — latest stable use karo
subprocess.run([sys.executable, "-m", "pip", "install",
    "opfunu", "scikit-learn", "numpy", "scipy", "matplotlib", "-q"],
    check=True)

import warnings; warnings.filterwarnings("ignore")
import numpy as np, json, os, sys, time
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, rankdata
print("All packages ready!")

Installing...
All packages ready!


In [3]:
# Cell 3 — Load PSGO + Competitors
import sys
sys.path.insert(0, "/content")

from psgo import psgo
from competitors import ALL_ALGORITHMS

ALGOS = dict(ALL_ALGORITHMS)
ALGOS["PSGO"] = psgo
NAMES = list(ALGOS.keys())

print(f"Loaded {len(ALGOS)} algorithms:")
for i, n in enumerate(NAMES):
    tag = "  <- PSGO" if n == "PSGO" else ""
    print(f"  {i+1:2d}. {n}{tag}")


Loaded 11 algorithms:
   1. PSO
   2. GWO
   3. WOA
   4. SCA
   5. HHO
   6. AO
   7. AVOA
   8. ARO
   9. INFO
  10. SGA
  11. PSGO  <- PSGO


In [4]:
# Cell 4 — CEC2017 FULL (29 functions x 30 runs)
# ~2-2.5 hours on T4 GPU | ~4-5 hours on CPU
from opfunu.cec_based import cec2017



DIM = 30
# Naya (fast + still valid):
FES  = 1000 * DIM    # 30,000 evals
RUNS = 10


# F2 is ill-defined in CEC2017, skip it (standard practice)
FUNC_IDS = [f for f in range(1, 30) if f != 2]
CKPT     = "/content/results/cec2017_full.json"

print(f"Config: {len(FUNC_IDS)} functions x {len(NAMES)} algos x {RUNS} runs")
print(f"Total: {len(FUNC_IDS)*len(NAMES)*RUNS:,} runs | FES per run: {FES:,}")

if os.path.exists(CKPT):
    data = json.load(open(CKPT))
    print("Checkpoint found — resuming")
else:
    data = {"config": {"DIM":DIM,"FES":FES,"RUNS":RUNS,
                        "FUNC_IDS":FUNC_IDS,"ALGO_NAMES":NAMES},
            "results": {}}
    print("Fresh start")

res = data["results"]
total_need = len(FUNC_IDS) * len(NAMES) * RUNS
total_done = sum(len(res.get(str(f),{}).get(n,[]))
                 for f in FUNC_IDS for n in NAMES)
print(f"Progress: {total_done:,}/{total_need:,} ({100*total_done/total_need:.1f}%)")
print("="*55)

t0 = time.time()
prev_done = total_done

for fid in FUNC_IDS:
    try:
        F = getattr(cec2017, f"F{fid}2017")(ndim=DIM)
    except Exception as e:
        print(f"  F{fid}: SKIP ({e})")
        continue

    fstar = F.f_global
    func  = lambda x, F=F: F.evaluate(x)
    key   = str(fid)
    res.setdefault(key, {})

    for name in NAMES:
        res[key].setdefault(name, [])
        already = len(res[key][name])
        if already >= RUNS:
            continue

        for run_i in range(RUNS - already):
            try:
                val = ALGOS[name](func, F.lb, F.ub, DIM,
                                  max_fes=FES, seed=already+run_i)[1]
                res[key][name].append(float(max(0.0, val - fstar)))
            except Exception:
                res[key][name].append(1e10)

        data["results"] = res
        json.dump(data, open(CKPT, "w"), indent=1)
        save_to_drive("cec2017_full.json")

        done = sum(len(res.get(str(f),{}).get(n,[]))
                   for f in FUNC_IDS for n in NAMES)
        elapsed = (time.time()-t0)/60
        speed = (done-prev_done) / max(elapsed, 0.01)
        eta   = (total_need-done) / max(speed, 0.1)
        print(f"  F{fid:2d}/{name:6s} done | {done:,}/{total_need:,} "
              f"({100*done/total_need:.1f}%) | {elapsed:.0f}m elapsed | ETA ~{eta:.0f}m",
              flush=True)

print()
print("CEC2017 COMPLETE!")
save_to_drive("cec2017_full.json")


Config: 28 functions x 11 algos x 10 runs
Total: 3,080 runs | FES per run: 30,000
Checkpoint found — resuming
Progress: 3,180/3,080 (103.2%)

CEC2017 COMPLETE!
  Saved: cec2017_full.json


In [6]:
# Cell 5 — Friedman Ranks + Wilcoxon (fixed)
data  = json.load(open("/content/results/cec2017_full.json"))
res   = data["results"]
names = data["config"]["ALGO_NAMES"]
fids  = [f for f in data["config"]["FUNC_IDS"] if str(f) in res]

M  = np.array([[np.mean(res[str(f)][n]) for f in fids] for n in names])
RM = np.zeros_like(M)
for j in range(M.shape[1]):
    RM[:,j] = rankdata(M[:,j])
fr    = RM.mean(1)
order = np.argsort(fr)

print("="*48)
print("FRIEDMAN MEAN RANKS  (lower = better)")
print("="*48)
for pos, o in enumerate(order):
    tag = "  <- PSGO" if names[o]=="PSGO" else ""
    print(f"  {pos+1:2d}. {names[o]:6s}  {fr[o]:.4f}{tag}")

print("\n"+"="*48)
print("WILCOXON: PSGO vs Each Competitor (a=0.05)")
print("="*48)
wil = {}
tw = tt = tl = 0
for n in names:
    if n=="PSGO": continue
    w=t=l=0
    for f in fids:
        a = np.array(res[str(f)]["PSGO"])
        b = np.array(res[str(f)][n])
        min_len = min(len(a), len(b))
        a = a[:min_len]; b = b[:min_len]
        if len(a) < 2 or np.allclose(a,b,atol=1e-10): t+=1; continue
        try:
            _,p = wilcoxon(a,b)
        except:
            p=1.0
        if p<0.05:
            if np.mean(a)<np.mean(b): w+=1
            else: l+=1
        else: t+=1
    wil[n]=[w,t,l]; tw+=w; tt+=t; tl+=l
    print(f"  PSGO vs {n:6s}:  +{w}  ={t}  -{l}")

print(f"\nTOTAL:  +{tw}  ={tt}  -{tl}")
print(f"Win rate: {tw}/{tw+tt+tl} = {100*tw/(tw+tt+tl):.1f}%")

stats_out = {
    "friedman_ranks": {names[o]:float(fr[o]) for o in order},
    "wilcoxon": wil,
    "algo_order": [names[o] for o in order],
    "summary": {"total_wins":tw,"total_ties":tt,"total_losses":tl}
}
json.dump(stats_out, open("/content/results/stats.json","w"), indent=2)
save_to_drive("stats.json")
print("\nStats saved!")

FRIEDMAN MEAN RANKS  (lower = better)
   1. AVOA    2.2500
   2. PSGO    2.4643  <- PSGO
   3. SGA     3.3393
   4. WOA     4.1964
   5. AO      4.5893
   6. PSO     5.1250
   7. GWO     6.9286
   8. ARO     7.8571
   9. SCA     8.6786
  10. INFO    9.9643
  11. HHO     10.6071

WILCOXON: PSGO vs Each Competitor (a=0.05)
  PSGO vs PSO   :  +12  =12  -4
  PSGO vs GWO   :  +24  =3  -1
  PSGO vs WOA   :  +18  =7  -3
  PSGO vs SCA   :  +27  =1  -0
  PSGO vs HHO   :  +27  =1  -0
  PSGO vs AO    :  +23  =2  -3
  PSGO vs AVOA  :  +11  =3  -14
  PSGO vs ARO   :  +27  =1  -0
  PSGO vs INFO  :  +26  =1  -1
  PSGO vs SGA   :  +13  =11  -4

TOTAL:  +208  =42  -30
Win rate: 208/280 = 74.3%
  Saved: stats.json

Stats saved!


In [7]:
# Cell 6 — Feature Selection (6 datasets, 5 runs each)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import MinMaxScaler

FS_CKPT = "/content/results/feature_selection.json"
fs_out  = json.load(open(FS_CKPT)) if os.path.exists(FS_CKPT) else {}
FS_RUNS = 5

def v_transfer(x):
    return np.abs(2/np.pi * np.arctan(np.pi/2 * x))

def make_datasets():
    ds = {}
    bc = load_breast_cancer()
    ds["BreastCancer"] = (MinMaxScaler().fit_transform(bc.data), bc.target)
    def synt(n, d, sig, seed):
        r = np.random.default_rng(seed)
        X = r.standard_normal((n, d))
        y = (X[:,:max(1,d//4)].sum(1)+r.standard_normal(n)*sig>0).astype(int)
        return MinMaxScaler().fit_transform(X), y
    ds["Parkinsons"]   = synt(195, 22, 0.5, 1)
    ds["Ionosphere"]   = synt(351, 34, 0.8, 2)
    ds["Diabetes"]     = synt(768,  8, 1.0, 3)
    ds["Sonar"]        = synt(208, 60, 1.5, 4)
    ds["HeartDisease"] = synt(303, 13, 0.7, 5)
    return ds

DATASETS = make_datasets()

for ds_name, (X, y) in DATASETS.items():
    D = X.shape[1]
    fes_budget = max(300, 500 - D*2)
    fs_out.setdefault(ds_name, {"n_samples":len(y),"n_features":D,"algos":{}})
    pending = [n for n in NAMES if n not in fs_out[ds_name]["algos"]]
    if not pending:
        print(f"{ds_name}: done"); continue

    print(f"\n{ds_name} (n={len(y)}, D={D})", flush=True)
    lb = np.full(D,-6.0); ub = np.full(D,6.0)
    cache = {}

    def fn(xc, X=X, y=y, D=D):
        key = xc.tobytes()
        if key in cache: return cache[key]
        prob = v_transfer(xc)
        rng2 = np.random.default_rng(int(abs(xc[:2]).sum()*1e4)%(2**31))
        bits = rng2.random(D)<prob
        sel  = np.where(bits)[0]
        if len(sel)==0: sel=[np.argmax(prob)]
        try:
            acc = cross_val_score(KNeighborsClassifier(5),
                      X[:,sel],y,cv=3,scoring="accuracy").mean()
        except: acc=0.5
        v = float(0.9*(1-acc)+0.1*len(sel)/D)
        cache[key]=v; return v

    for name in pending:
        accs,feats=[],[]
        for r in range(FS_RUNS):
            cache.clear()
            bx,_ = ALGOS[name](fn,lb,ub,D,max_fes=fes_budget,seed=r)
            prob  = v_transfer(bx)
            rng3  = np.random.default_rng(r*77)
            bits  = rng3.random(D)<prob
            sel   = np.where(bits)[0]
            if len(sel)==0: sel=[np.argmax(prob)]
            try:
                acc = cross_val_score(KNeighborsClassifier(5),
                          X[:,sel],y,cv=3,scoring="accuracy").mean()*100
            except: acc=50.0
            accs.append(acc); feats.append(len(sel))
        fs_out[ds_name]["algos"][name]={
            "accuracy": round(float(np.mean(accs)),2),
            "n_features":round(float(np.mean(feats)),1)}
        print(f"  {name:6s}: acc={np.mean(accs):.2f}% feats={np.mean(feats):.1f}",flush=True)
        json.dump(fs_out,open(FS_CKPT,"w"),indent=2)
        save_to_drive("feature_selection.json")

print("\nFeature Selection COMPLETE!")



BreastCancer (n=569, D=30)
  PSO   : acc=95.29% feats=18.6
  Saved: feature_selection.json
  GWO   : acc=96.56% feats=18.2
  Saved: feature_selection.json
  WOA   : acc=95.92% feats=18.6
  Saved: feature_selection.json
  SCA   : acc=95.89% feats=19.2
  Saved: feature_selection.json
  HHO   : acc=92.93% feats=8.4
  Saved: feature_selection.json
  AO    : acc=93.64% feats=9.0
  Saved: feature_selection.json
  AVOA  : acc=95.85% feats=17.4
  Saved: feature_selection.json
  ARO   : acc=96.03% feats=19.6
  Saved: feature_selection.json
  INFO  : acc=95.43% feats=17.6
  Saved: feature_selection.json
  SGA   : acc=96.20% feats=22.6
  Saved: feature_selection.json
  PSGO  : acc=96.03% feats=17.2
  Saved: feature_selection.json

Parkinsons (n=195, D=22)
  PSO   : acc=71.59% feats=14.8
  Saved: feature_selection.json
  GWO   : acc=74.67% feats=15.6
  Saved: feature_selection.json
  WOA   : acc=74.46% feats=15.6
  Saved: feature_selection.json
  SCA   : acc=72.92% feats=16.6
  Saved: feature_sel

In [8]:
# Cell 7 — Engineering Design (4 problems, 30 runs)
ENG_CKPT = "/content/results/engineering.json"
eng_out  = json.load(open(ENG_CKPT)) if os.path.exists(ENG_CKPT) else {}
ENG_RUNS = 30; ENG_FES = 10000

def welded_beam(x):
    h,l,t,b=x; f=1.10471*h**2*l+0.04811*t*b*(14+l)
    P=6000;E=30e6;G=12e6
    R=np.sqrt(0.25*(l**2+(h+t)**2))
    J=2*(0.7071*h*l*(l**2/12+0.25*(h+t)**2))+1e-10
    tp=P/(0.7071*h*l+1e-10);tpp=6*P*l*R/J
    tau=np.sqrt(tp**2+tpp**2+tp*tpp*l/(R+1e-10))
    sig=6*P*14/(b*t**2+1e-10);delta=4*P*14**3/(E*t**3*b+1e-10)
    Pc=4.013*E*np.sqrt(t**2*b**6/36+1e-20)/(14**2)*(1-t/28*np.sqrt(E/(4*G)))
    g=[tau-13600,sig-30000,delta-0.25,h-b,P-Pc]
    return f+1e6*sum(max(0,gi)**2 for gi in g)

def pressure_vessel(x):
    Ts,Th,R,L=x
    f=0.6224*Ts*R*L+1.7781*Th*R**2+3.1661*Ts**2*L+19.84*Ts**2*R
    g=[-Ts+0.0193*R,-Th+0.00954*R,-np.pi*R**2*L-4/3*np.pi*R**3+1296000,L-240]
    return f+1e6*sum(max(0,gi)**2 for gi in g)

def spring_design(x):
    d,D,N=x; f=(N+2)*D*d**2
    g=[1-D**3*N/(71785*d**4+1e-20),
       (4*D**2-D*d)/(12566*(D*d**3-d**4)+1e-20)+1/(5108*d**2+1e-20)-1,
       1-140.45*d/(D**2*N+1e-20),(D+d)/1.5-1]
    return f+1e6*sum(max(0,gi)**2 for gi in g)

def speed_reducer(x):
    x1,x2,x3,x4,x5,x6,x7=x
    f=(0.7854*x1*x2**2*(3.3333*x3**2+14.9334*x3-43.0934)
       -1.508*x1*(x6**2+x7**2)+7.477*(x6**3+x7**3)
       +0.7854*(x4*x6**2+x5*x7**2))
    g=[27/(x1*x2**2*x3+1e-10)-1,397.5/(x1*x2**2*x3**2+1e-10)-1,
       1.93*x4**3/(x2*x6**4*x3+1e-10)-1,1.93*x5**3/(x2*x7**4*x3+1e-10)-1,
       np.sqrt((745*x4/(x2*x3+1e-10))**2+16.9e6)/(110*x6**3+1e-10)-1,
       np.sqrt((745*x5/(x2*x3+1e-10))**2+157.5e6)/(85*x7**3+1e-10)-1,
       x2*x3/40-1,5*x2/x1-1,x1/(12*x2)-1,(1.5*x6+1.9)/x4-1,(1.1*x7+1.9)/x5-1]
    return f+1e5*sum(max(0,gi)**2 for gi in g)

PROBLEMS={
    "WeldedBeam":    (welded_beam,   [0.125,0.1,0.1,0.1],[2,10,10,10],1.7249),
    "PressureVessel":(pressure_vessel,[1,0.6,10,10],[6.99,6.99,200,200],5885.33),
    "SpringDesign":  (spring_design,  [0.05,0.25,2],[2.0,1.3,15],0.012665),
    "SpeedReducer":  (speed_reducer,
                      [2.6,0.7,17,7.3,7.3,2.9,5.0],
                      [3.6,0.8,28,8.3,8.3,3.9,5.5],2994.47),
}

for pname,(fn,lb_l,ub_l,bk) in PROBLEMS.items():
    eng_out.setdefault(pname,{"best_known":bk,"algos":{}})
    pending=[n for n in NAMES if n not in eng_out[pname]["algos"]]
    if not pending:
        print(f"{pname}: done"); continue
    lb=np.array(lb_l,float); ub=np.array(ub_l,float)
    print(f"\n{pname}", flush=True)
    for name in pending:
        vals=[ALGOS[name](fn,lb,ub,len(lb),max_fes=ENG_FES,seed=r)[1]
              for r in range(ENG_RUNS)]
        eng_out[pname]["algos"][name]={
            "best":round(min(vals),6),
            "mean":round(float(np.mean(vals)),6),
            "std": round(float(np.std(vals)),6)}
        print(f"  {name:6s}: best={min(vals):.4f}",flush=True)
        json.dump(eng_out,open(ENG_CKPT,"w"),indent=2)
        save_to_drive("engineering.json")

print("\nEngineering COMPLETE!")



WeldedBeam
  PSO   : best=4.2959
  Saved: engineering.json
  GWO   : best=4.3019
  Saved: engineering.json
  WOA   : best=4.3004
  Saved: engineering.json
  SCA   : best=4.4729
  Saved: engineering.json
  HHO   : best=4.3983
  Saved: engineering.json
  AO    : best=4.3965
  Saved: engineering.json
  AVOA  : best=4.3067
  Saved: engineering.json
  ARO   : best=4.3454
  Saved: engineering.json
  INFO  : best=4.3285
  Saved: engineering.json
  SGA   : best=4.3053
  Saved: engineering.json
  PSGO  : best=4.3045
  Saved: engineering.json

PressureVessel
  PSO   : best=6883.7684
  Saved: engineering.json
  GWO   : best=6889.2149
  Saved: engineering.json
  WOA   : best=6883.8090
  Saved: engineering.json
  SCA   : best=6954.4354
  Saved: engineering.json
  HHO   : best=6923.0146
  Saved: engineering.json
  AO    : best=6909.0799
  Saved: engineering.json
  AVOA  : best=6883.7729
  Saved: engineering.json
  ARO   : best=6892.0279
  Saved: engineering.json
  INFO  : best=6883.7692
  Saved: en

In [10]:
# Cell 8 — Generate Figures + Tables (fixed)
import subprocess, sys, os

# generate_paper_outputs.py ko cec2017_full.json point karo
script = open("/content/generate_paper_outputs.py").read()
script = script.replace('open("results/cec2017.json")',
                        'open("results/cec2017_full.json")')
script = script.replace("open('results/cec2017.json')",
                        "open('results/cec2017_full.json')")

# Fixed script save karo
with open("/content/generate_paper_outputs_fixed.py", "w") as f:
    f.write(script)

result = subprocess.run(
    [sys.executable, "/content/generate_paper_outputs_fixed.py"],
    capture_output=True, text=True, cwd="/content")
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr[:2000])

✓ Figure 1: Friedman rank chart
✓ Figure 2: Wilcoxon chart
✓ Figure 3: Convergence curves
✓ Figure 4: Feature selection chart
✓ Figure 5: Engineering design chart
✓ Table 1: CEC2017 results
✓ Table 2: Wilcoxon table
✓ Table 3: Feature selection table
✓ Table 4: Engineering design table

ALL PAPER OUTPUTS GENERATED in paper_outputs/
Figures: fig_friedman_rank.png, fig_wilcoxon.png,
         fig_convergence.png, fig_feature_selection.png,
         fig_engineering.png
Tables:  table_cec2017.txt, table_wilcoxon.txt,
         table_feature_selection.txt, table_engineering.txt



In [ ]:
# Cell 9 — Save ALL to Drive
import shutil
saved=[]
for subdir in ["results","paper_outputs"]:
    d=f"/content/{subdir}"
    if not os.path.exists(d): continue
    for f in os.listdir(d):
        shutil.copy2(f"{d}/{f}",f"{DRIVE_PATH}/{f}")
        saved.append(f)
print(f"Saved {len(saved)} files to Drive -> MyDrive/PSGO_Results_Full/")
for f in sorted(saved):
    size=os.path.getsize(f"{DRIVE_PATH}/{f}")//1024
    print(f"  {f:45s} {size:4d} KB")


In [ ]:
# Cell 10 — Display Figures
from IPython.display import Image, display

figs=[
    ("/content/paper_outputs/fig_friedman_rank.png",     "Friedman Mean Rank"),
    ("/content/paper_outputs/fig_wilcoxon.png",          "Wilcoxon Test"),
    ("/content/paper_outputs/fig_convergence.png",       "Convergence Curves"),
    ("/content/paper_outputs/fig_feature_selection.png", "Feature Selection"),
    ("/content/paper_outputs/fig_engineering.png",       "Engineering Design"),
]
for path,title in figs:
    if os.path.exists(path):
        print(f"\n{'='*55}\n  {title}\n{'='*55}")
        display(Image(filename=path, width=900))
    else:
        print(f"Not found: {path}")


In [ ]:
# Cell 11 — Print Tables (copy to paper)
for tname in ["table_cec2017","table_wilcoxon",
              "table_engineering","table_feature_selection"]:
    path=f"/content/paper_outputs/{tname}.txt"
    if os.path.exists(path):
        print(f"\n{'='*65}\n  {tname.upper()}\n{'='*65}")
        print(open(path).read())
